# Module 4: Online evals and Automations
> ~25 min

In Module 3, Engine found the agent's worst failure on its own. This module builds the same kind of watch by hand: two LLM judges score every new conversation, and two **automations** act on those scores. Good turns land in a golden dataset for Module 5, and bad turns land in a review queue for a person.

## Setup
This cell loads your keys and finds the tracing project behind your agent's `development` environment. Every judge and automation in this module attaches to that project.

> **Requires:** this workspace's LLM Gateway can serve the judge model (`JUDGE_MODEL_ID`, through a provider key added to the Gateway or Gateway Credits); LangSmith runs both judges there. **Without it:** the judges and automations are still created but no scores appear; set `JUDGE_MODEL_ID` to a model your Gateway lists and rerun from section 1.

On the older project setup (`LANGSMITH_PROJECT` in `.env`), the cell uses that project instead.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import asyncio
from datetime import datetime, timezone

from langsmith import Client

from utils.config import GOLDEN_DATASET, REVIEW_QUEUE, environment_project
from utils.langsmith_rules import queue_url, upsert_automation, upsert_judge, upsert_thread_judge
from utils.notebook import trace_link
from utils.traffic import generate_traffic

JUDGE_MODEL_ID = "openai/gpt-5.6-luna"  # a model this workspace's LLM Gateway serves
client = Client()
project = environment_project(client)
start = datetime.now(timezone.utc)

print("Rules attach to:", project.project_name)
trace_link(project.url, "Tracing project")

## 1. An online judge for every turn
An **online evaluator** is an LLM judge that LangSmith runs on new runs as they arrive; it writes its score to the run as **feedback**. This one scores each agent turn from 0 to 1 under the key `correctness`, and the filter `eq(is_root, true)` keeps it to one score per turn instead of one per model and tool call.

In [ ]:
CORRECTNESS_PROMPT = (
    "You grade one turn of a client research assistant at a wealth management firm. "
    "You see the user's request and the agent's full run, including its tool calls and results. "
    "Score 1 when the final reply answers the request with facts taken from the tool results. "
    "Score 0 when the reply does not give the user what they asked for, invents facts, "
    "or ends without an answer. Use values in between for partial answers."
)
CORRECTNESS_SCHEMA = {
    "title": "extract",
    "description": "Grade one agent turn.",
    "type": "object",
    "properties": {
        "comment": {"type": "string", "description": "Reasoning for the score"},
        "correctness": {"type": "number", "minimum": 0, "maximum": 1,
                        "description": "How well the reply answers the request, from 0 to 1."},
    },
    "required": ["correctness"],
}

judge = upsert_judge(client, project, name="correctness", prompt=CORRECTNESS_PROMPT,
                     schema=CORRECTNESS_SCHEMA, filter="eq(is_root, true)", model_id=JUDGE_MODEL_ID)
print("Judge:", judge.name, judge.id)
trace_link(judge.url, "Online Evaluators")

Open the link and click `correctness`. **Feedback configuration** shows a **Continuous** key from 0 to 1; `comment` holds the judge's reasoning, not a second score.

<img src="../images/configure_online_eval.png" alt="Online evaluator configuration showing the feedback configuration and filter" style="width: auto; max-height: 360px; border-radius: 8px;">

## 2. A judge for whole conversations
A **thread evaluator** waits until a conversation has been idle for a set time, then judges all of its turns at once. It catches what a per-turn judge misses: a conversation can sound helpful on every turn and still end without what the user came for.

In [ ]:
RESOLVED_PROMPT = (
    "You read a whole conversation between a relationship manager and a client research assistant. "
    "Answer true if, by the end, the user got what they asked for. "
    "Answer false if the assistant gave up, asked the user for something it should have found, "
    "or never answered."
)
RESOLVED_SCHEMA = {
    "title": "extract",
    "description": "Judge one conversation.",
    "type": "object",
    "properties": {
        "comment": {"type": "string", "description": "Reasoning for the score"},
        "thread_resolved": {"type": "boolean", "description": "Did the user get what they asked for?"},
    },
    "required": ["thread_resolved"],
}

thread_judge = upsert_thread_judge(client, project, name="thread_resolved", prompt=RESOLVED_PROMPT,
                                   schema=RESOLVED_SCHEMA, idle_seconds=120, model_id=JUDGE_MODEL_ID)
print("Thread judge:", thread_judge.name, "| idle time: 2 min")
trace_link(thread_judge.url, "Online Evaluators")

The same tab now lists `thread_resolved` with **Source: Threads**. Its idle time is a project setting: 2 minutes is the minimum, and it applies to every thread evaluator on this project.

<img src="../images/evals_tab.png" alt="Online Evaluators tab listing correctness and thread_resolved" style="width: auto; max-height: 360px; border-radius: 8px;">

> Give a thread judge its own feedback key: in the **Feedback** column, a thread score and a per-turn score under the same key look identical.

## 3. Automation: good turns become golden examples
An **automation** is a saved filter, a sampling rate and one action. Each automation polls on its own schedule, so it can act on feedback a judge wrote a minute earlier. This one copies every first turn scored `correctness >= 0.9` into the dataset `client-research-golden`.

In [ ]:
if not client.has_dataset(dataset_name=GOLDEN_DATASET):
    client.create_dataset(GOLDEN_DATASET, description="First turns the correctness judge scored 0.9 or higher.")
golden = client.read_dataset(dataset_name=GOLDEN_DATASET)

# first turns only: later turns arrive without the conversation before them
GOLDEN_FILTER = (
    'and(eq(is_root, true), eq(feedback_key, "correctness"), gte(feedback_score, 0.9), '
    'eq(metadata_key, "turn"), eq(metadata_value, "1"))'
)
golden_rule = upsert_automation(client, project, name="golden-correctness",
                                filter=GOLDEN_FILTER, dataset_id=golden.id)
print("golden-correctness →", golden.name)
trace_link(golden_rule.url, "Automations")

Open the link and click `golden-correctness`. The form shows **Item Type: Runs**, the filter, the sampling rate at 100%, and the **Add to Dataset** action. The form can also create a dataset inline; the cell creates it first because the API needs its ID.

<img src="../images/TODO-04-automation-form.png" alt="Automation form for golden-correctness with its filter, sampling rate and Add to Dataset action" style="width: auto; max-height: 360px; border-radius: 8px;">

## 4. Automation: bad turns go to a person
The same filter with the threshold flipped sends low scores to an **annotation queue**, where a reviewer reads the trajectory and records a verdict. The queue's **rubric** sets the fields the reviewer fills in; here it is one categorical key, `failure_type`.

In [ ]:
FAILURE_TYPES = ["tool_error", "wrong_answer", "out_of_scope", "judge_disagrees"]
client.create_feedback_config(
    "failure_type",
    feedback_config={
        "type": "categorical",
        "categories": [{"value": i, "label": label} for i, label in enumerate(FAILURE_TYPES)],
    },
)

queue = next(client.list_annotation_queues(name=REVIEW_QUEUE), None)
if queue is None:
    queue = client.create_annotation_queue(
        name=REVIEW_QUEUE,
        rubric_instructions="Read the trajectory, then pick what went wrong in this turn.",
        rubric_items=[{"feedback_key": "failure_type", "is_required": True}],
    )
print("Queue:", queue.name)
trace_link(queue_url(project.url, queue.id), "Review queue")

In [ ]:
REVIEW_FILTER = 'and(eq(is_root, true), eq(feedback_key, "correctness"), lt(feedback_score, 0.5))'
review_rule = upsert_automation(client, project, name="review-low-correctness",
                                filter=REVIEW_FILTER, queue_id=queue.id)
print("review-low-correctness →", queue.name)
!langsmith evaluator rule list --project-id {project.project_id} --all

The **Actions** column is this module in four rows: two rules write feedback (`evaluator`), and two read it (`add_to_dataset`, `add_to_annotation_queue`).

## 5. Send new conversations
Judges and automations act only on runs that arrive after they exist. This cell replays the seed conversations from Module 2 through the agent; it takes about 3 minutes.

In [ ]:
thread_ids = generate_traffic()
print("Threads sent:", len(thread_ids))
trace_link(project.url, "Tracing project")

`correctness` scores appear a minute or two after each turn. `thread_resolved` scores appear 2 minutes after a thread's last turn. Each automation acts within about 4 minutes of a score landing.

## 6. Read the scores
Querying runs with the two automations' own filters shows exactly which turns each one will act on. The cell checks every 30 seconds, for up to 10 minutes, until both buckets have runs.

In [ ]:
async def matching_runs(run_filter):
    runs = []
    async for run in client.runs.query(
        project_ids=[project.project_id],
        filter=run_filter,
        min_start_time=start,
        selects=["ID", "INPUTS_PREVIEW"],
    ):
        runs.append(run)
    return runs

for attempt in range(20):
    golden_runs = await matching_runs(GOLDEN_FILTER)
    review_runs = await matching_runs(REVIEW_FILTER)
    if golden_runs and review_runs:
        break
    await asyncio.sleep(30)

print("First turns scored >= 0.9:", len(golden_runs))
print("Turns scored < 0.5:", len(review_runs))
for run in review_runs[:3]:
    print("  low:", run.inputs_preview)

The low scores cluster on client-lookup questions, the same failure Engine filed in Module 3. In the UI, **Threads** shows `thread_resolved` false on those conversations.

<img src="../images/online_evals.png" alt="Runs with correctness feedback in the Feedback column" style="width: auto; max-height: 360px; border-radius: 8px;">

> If both counts are 0, the judges have no model to call: see the **Requires** note in Setup.

## 7. What the automations did
Each automation copies or queues a run on its next poll after the score lands. The cell waits 4 minutes for that, then counts the dataset's examples and the queue's items.

In [ ]:
await asyncio.sleep(240)  # automations act within ~4 min of a score
examples = list(client.list_examples(dataset_id=golden.id))
queued = await client.annotation_queues.items.retrieve_count(str(queue.id), status="all")

print("Golden examples:", len(examples))
print("Queued for review:", queued.count)
if examples:
    print("First example input:", str(examples[0].inputs)[:200])
trace_link(golden.url, "Golden dataset")

In **Automations**, click **Logs** on `golden-correctness`: each run it processed is listed with a link to the run. Module 5 runs experiments on this dataset.

<img src="../images/TODO-04-automation-logs.png" alt="Logs for the golden-correctness automation listing processed runs" style="width: auto; max-height: 360px; border-radius: 8px;">

## 8. Review a queued turn
A queued run opens in the trajectory view. The reviewer reads it, picks a `failure_type`, and clicks **Done**, which moves the queue to the next item.

Open the **Review queue** link from section 4. For two items, read the failing turn's tool calls, pick a `failure_type`, add a line under **Reviewer Notes**, and click **Done**.

<img src="../images/annotation_queue.png" alt="Annotation queue review pane with the trajectory and the failure_type rubric" style="width: auto; max-height: 360px; border-radius: 8px;">

Your `failure_type` now sits on the run next to the judge's `correctness`, as human feedback. **Add to Dataset** in the same pane saves a reviewed run as an example by hand.

## Recap
| Step | Where it lives |
|---|---|
| Per-turn judge | **Online Evaluators** tab · `upsert_judge` (REST `POST /runs/rules`) |
| Thread judge | **Online Evaluators** → **Source: Threads** · `upsert_thread_judge` |
| Automation | **Automations** → **+ New** → **New Automation** · `upsert_automation` |
| List rules | `langsmith evaluator rule list --all` |
| Read scores | `client.runs.query(filter=...)` |
| Review queue | **Annotation Queues** · `client.create_annotation_queue(rubric_items=...)` |
| Queue count | `client.annotation_queues.items.retrieve_count` |

Your `correctness` scores also feed Engine's next scan.

**Next:** Module 5 — Offline evals

**Docs:** [LLM-as-a-judge online evaluators](https://docs.langchain.com/langsmith/online-evaluations-llm-as-judge) · [Multi-turn evaluators](https://docs.langchain.com/langsmith/online-evaluations-multi-turn) · [Automation rules](https://docs.langchain.com/langsmith/rules) · [Annotation queues](https://docs.langchain.com/langsmith/annotation-queues) · [Annotation queues (SDK)](https://docs.langchain.com/langsmith/annotation-queues-sdk) · [Trace query syntax](https://docs.langchain.com/langsmith/trace-query-syntax)